In [ ]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_rnafm_token_nt450_ribotricer_openprot_transcript_5to3_v2.py \
  --input_table /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_pos.pkl \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_pos \
  --local_model_dir /disk3/hejp/00_thesis/4_predict_translation/model/rnafm \
  --model_name rna-fm \
  --run_region all \
  --upstream_col upstream_seq_50nt \
  --orf_col orf_seq \
  --downstream_col downstream_seq_50nt \
  --id_col sample_id \
  --label_col label_raw \
  --min_aa 30 \
  --max_aa 150 \
  --dna_source coding \
  --batch_size 1 \
  --too_long skip \
  --device cuda

In [ ]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_rnafm_token_nt450_ribotricer_openprot_transcript_5to3_v2.py \
  --input_table /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_neg.pkl \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_neg \
  --local_model_dir /disk3/hejp/00_thesis/4_predict_translation/model/rnafm \
  --model_name rna-fm \
  --run_region all \
  --upstream_col upstream_seq_50nt \
  --orf_col orf_seq \
  --downstream_col downstream_seq_50nt \
  --id_col sample_id \
  --label_col label_raw \
  --min_aa 30 \
  --max_aa 150 \
  --dna_source coding \
  --batch_size 1 \
  --too_long skip \
  --device cuda

In [ ]:
###合并pos和neg###
import pandas as pd

# 输入文件
train_pos_pkl = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_pos.pkl"
train_neg_pkl = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_neg.pkl"

train_pos_csv = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_pos.csv.gz"
train_neg_csv = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_table_neg.csv.gz"

# 输出文件
out_pkl = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_pos_neg_merged.pkl"
out_csv = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_pos_neg_merged.csv.gz"

# 读取 pkl
pos = pd.read_pickle(train_pos_pkl)
neg = pd.read_pickle(train_neg_pkl)
cols = pos.columns.tolist()
cols[-2] = "protein_len_a"
pos.columns = cols
# 合并
train_all = pd.concat([pos, neg], axis=0, ignore_index=True)

print("pos:", pos.shape)
print("neg:", neg.shape)
print("merged:", train_all.shape)

print(train_all["label_raw"].value_counts(dropna=False))
print(train_all["label_id"].value_counts(dropna=False))

display(train_all.head())

# 保存 pkl 和 csv.gz
train_all.to_pickle(out_pkl)
train_all.to_csv(out_csv, index=False, compression="gzip")

print("saved:")
print(out_pkl)
print(out_csv)

In [ ]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_esm2_token_embedding_nt450_ribotricer_openprot.py \
  --DATA_PATH /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_pos_neg_merged.pkl \
  --SAVE_DIR /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_esm2_token_pos_neg

In [ ]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/src_v3/src_rnafm_self_cross_attention_v3/rnafm_10run_strict_v4/build_translation_len_matched_10runs.py \
  --metadata_csv /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_pos_neg_merged.csv.gz \
  --rnafm_token_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_pos_neg \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/rnafm_only/ \
  --prefix translation_strict_lenmatched \
  --match_mode strict_pair \
  --n_runs 10 \
  --seed 42

In [ ]:
###优化使用Transformer enhanced position 用ROC筛选###
!python /disk3/hejp/00_thesis/4_predict_translation/src/src_v3/src_rnafm_self_cross_attention_v3/rnafm_10run_strict_v4/stage1_rnafm_trans_position_v1_roc/run_10runs_enhanced_position.py \
  --train_script /disk3/hejp/00_thesis/4_predict_translation/src/src_v3/src_rnafm_self_cross_attention_v3/rnafm_10run_strict_v4/stage1_rnafm_trans_position_v1_roc/train_rnafm_self_attention_enhanced_position.py \
  --dataset_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/rnafm_only \
  --dataset_prefix translation_strict_lenmatched \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/rnafm_self_attention_strict_10runs/afteroptimize/enhanced_position_ffmult2_wd5e4_10runs_roc \
  --run_name_prefix rnafm_self_attention_enhanced_position_select_strict_roc \
  --position_mode enhanced \
  --monitor_metric roc_auc \
  --start_run 0 \
  --n_runs 10 \
  --seed 42 \
  --batch_size 64 \
  --epochs 50 \
  --num_heads 8 \
  --num_layers 2 \
  --dropout 0.2 \
  --lr 1e-4 \
  --weight_decay 5e-4 \
  --ff_mult 2 \
  --branch_hidden_dim 256 \
  --fusion_hidden_dim 256 \
  --patience 10 \
  --grad_clip 1.0 \
  --num_workers 2 \
  --skip_existing